# 🚀 FabrIQ Fabric Defect Detection - YOLO Training (Google Colab)

This notebook trains YOLOv8 models for fabric defect detection using Google Colab's free GPU.

## 📋 Steps:
1. Setup environment and install dependencies
2. Upload and organize dataset
3. Create annotations (for detection)
4. Train YOLO model
5. Evaluate and download results

**Note**: This notebook is designed to work with VS Code's Colab extension. Make sure you have the extension installed and connected to Google Colab.


## Step 1: Install Dependencies


In [2]:
# Install required packages
%pip install ultralytics torch torchvision opencv-python pillow numpy pyyaml tqdm -q

# Verify installation
import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Kaggle: install compatible versions (run this first, then restart if prompted)
%pip install --no-cache-dir "ultralytics==8.0.196" "numpy==1.26.4" "opencv-python==4.11.0.86" -q

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Allow ultralytics weights to load with PyTorch 2.6 (weights_only default change)
from ultralytics.nn.tasks import DetectionModel
import torch.serialization as ts

# Trust ultralytics DetectionModel when loading .pt weights
try:
    ts.add_safe_globals([DetectionModel])
    print("✅ Added DetectionModel to torch safe globals")
except Exception as e:
    print("⚠️ Could not add safe globals:", e)



In [ ]:
# Kaggle: train detection using already-uploaded YOLO dataset
from pathlib import Path
import shutil
import cv2
import numpy as np
from ultralytics import YOLO
import yaml

# Input YOLO dataset (already uploaded)
INPUT_ROOT = Path('/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset')
WORK_ROOT = Path('/kaggle/working/FabrIQ_YOLO_Detection_Dataset')

# Verify structure
for p in [INPUT_ROOT / 'train' / 'images', INPUT_ROOT / 'train' / 'labels', INPUT_ROOT / 'val' / 'images', INPUT_ROOT / 'val' / 'labels']:
    assert p.exists(), f"Missing path: {p}"

# Copy to working dir once (skip if exists)
if not WORK_ROOT.exists():
    shutil.copytree(INPUT_ROOT, WORK_ROOT)
    print(f"✅ Copied dataset to {WORK_ROOT}")
else:
    print(f"ℹ️ Working copy already exists: {WORK_ROOT}")

# Classes
CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

# Clean any corrupt JPGs (re-encode; if still bad, delete image+label)
def is_image_ok(path: Path) -> bool:
    try:
        data = np.fromfile(path, dtype=np.uint8)
        if data.size == 0:
            return False
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img is None:
            return False
        cv2.imencode(path.suffix, img)[1].tofile(path)
        return True
    except Exception:
        return False

bad = 0
for split in ['train', 'val']:
    img_dir = WORK_ROOT / split / 'images'
    lbl_dir = WORK_ROOT / split / 'labels'
    for img_path in img_dir.glob('*.jpg'):
        if not is_image_ok(img_path):
            bad += 1
            (lbl_dir / f"{img_path.stem}.txt").unlink(missing_ok=True)
            img_path.unlink(missing_ok=True)
print(f"🧹 Cleaned corrupt images: removed {bad} files (if any)")

# Write data.yaml
yaml_path = Path('/kaggle/working/fabriq_detection_data.yaml')
data_config = {
    'path': str(WORK_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CLASSES),
    'names': {i: cls for i, cls in enumerate(CLASSES)}
}
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
print(f"✅ data.yaml: {yaml_path}")

# Train
model = YOLO('yolov8n.pt')
results = model.train(
    data=str(yaml_path),
    epochs=5,          # adjust as needed
    imgsz=640,
    batch=16,
    device=0,
    workers=0,         # safer on Kaggle
    amp=False,         # avoid mixed-precision issues if any
    project='/kaggle/working/runs/detect',
    name='fabriq_detection',
    exist_ok=True,
    patience=50,
    save=True,
    plots=True
)

print("\n✅ Training complete")
print("Results:", results.save_dir)



## Step 2: Upload Dataset

**Option A: Upload from your computer**


**Option B: Use Google Drive**


## Step 3: Organize Dataset


In [5]:
# Prepare classification dataset
from pathlib import Path
import shutil

yolo_dataset = Path('/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset')
source_dataset = Path('/kaggle/input/fabric-final-dataset/FabrIQ_Final_Dataset')

CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

for split in ['train', 'val']:
    source_dir = source_dataset / split / 'images'
    
    # Create class folders
    for class_name in CLASSES:
        class_folder = yolo_dataset / split / class_name.replace(' ', '_')
        class_folder.mkdir(parents=True, exist_ok=True)
    
    # Copy images to class folders
    count = 0
    for img_file in source_dir.glob("*.jpg"):
        filename = img_file.stem
        parts = filename.split('_')
        
        if len(parts) >= 2 and parts[0] == 'FabrIQ':
            class_parts = parts[1:-1]
            class_name = '_'.join(class_parts)
            class_name_with_spaces = class_name.replace('_', ' ')
            
            if class_name_with_spaces in CLASSES:
                dest_folder = yolo_dataset / split / class_name
                shutil.copy2(img_file, dest_folder / img_file.name)
                count += 1
    
    print(f"✅ {split}: {count} images organized")

print(f"\n📁 Classification dataset ready at: {yolo_dataset}")


OSError: [Errno 30] Read-only file system: '/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset/train/bad_needle_line'

In [6]:
"""
YOLOv8 Object Detection Training Script for FabrIQ Fabric Defect Detection
This script trains a detection model that can predict bounding boxes around defects.
Note: Requires YOLO-format annotation files (.txt) with bounding boxes.
"""

import os
from pathlib import Path
from ultralytics import YOLO
import yaml

# Configuration
DATASET_PATH = Path("/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset")
MODEL_TYPE = "yolov8n.pt"  # Detection model
# Auto-detect device
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":
        EPOCHS = 30
        IMG_SIZE = 416  # Smaller for CPU
        BATCH_SIZE = 4
    else:
        EPOCHS = 5
        IMG_SIZE = 640
        BATCH_SIZE = 16
except:
    DEVICE = "cpu"
    EPOCHS = 10
    IMG_SIZE = 416
    BATCH_SIZE = 4

# 20 Target Classes
CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

def create_data_yaml():
    """Create data.yaml configuration file for YOLO detection"""
    dataset_path = Path("FabrIQ_YOLO_Detection_Dataset")
    
    data_config = {
        'path': str(dataset_path.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'nc': len(CLASSES),
        'names': {i: cls for i, cls in enumerate(CLASSES)}
    }
    
    yaml_path = Path("fabriq_detection_data.yaml")
    with open(yaml_path, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
    
    print(f"✅ Created data configuration: {yaml_path}")
    return yaml_path

def prepare_detection_dataset():
    """Prepare dataset structure for YOLO object detection"""
    print("📁 Preparing dataset structure for YOLO object detection...")
    
    yolo_dataset = Path("/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset/")
    
    # Create directory structure
    for split in ['train', 'val']:
        (yolo_dataset / split / 'images').mkdir(parents=True, exist_ok=True)
        (yolo_dataset / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Copy images
    for split in ['train', 'val']:
        source_dir = DATASET_PATH / split / 'images'
        
        if not source_dir.exists():
            print(f"⚠️ Warning: {source_dir} does not exist!")
            continue
        
        print(f"   Copying {split} images...")
        image_count = 0
        
        for img_file in source_dir.glob("*.jpg"):
            dest_img = yolo_dataset / split / 'images' / img_file.name
            import shutil
            shutil.copy2(img_file, dest_img)
            image_count += 1
        
        print(f"   ✅ {split}: {image_count} images copied")
        print(f"   ⚠️ Note: You need to create annotation files (.txt) in {yolo_dataset / split / 'labels'}/")
        print(f"      Format: class_id center_x center_y width height (normalized 0-1)")
    
    return yolo_dataset

def train_detection_model():
    """Train YOLOv8 object detection model"""
    print("\n🚀 Starting YOLOv8 Object Detection Training...")
    
    # Prepare dataset
    dataset_path = prepare_detection_dataset()
    
    # Create data.yaml
    yaml_path = create_data_yaml()
    
    # Check if annotation files exist
    train_labels = list((dataset_path / 'train' / 'labels').glob('*.txt'))
    val_labels = list((dataset_path / 'val' / 'labels').glob('*.txt'))
    
    if len(train_labels) == 0:
        print("\n⚠️ WARNING: No annotation files found!")
        print("   For object detection, you need bounding box annotations.")
        print("   Run create_annotations.py or manually annotate images using tools like:")
        print("   - LabelImg (https://github.com/tzutalin/labelImg)")
        print("   - Roboflow (https://roboflow.com/)")
        print("   - CVAT (https://cvat.org/)")
        return None, None
    
    print(f"\n   Found {len(train_labels)} train annotations and {len(val_labels)} val annotations")
    
    # Initialize model
    print(f"\n📦 Loading model: {MODEL_TYPE}")
    model = YOLO(MODEL_TYPE)
    
    # Train the model
    print(f"\n🎯 Training Configuration:")
    print(f"   Dataset: {dataset_path}")
    print(f"   Config: {yaml_path}")
    print(f"   Epochs: {EPOCHS}")
    print(f"   Image Size: {IMG_SIZE}")
    print(f"   Batch Size: {BATCH_SIZE}")
    print(f"   Device: {DEVICE}")
    print(f"   Classes: {len(CLASSES)}")
    
    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project="runs/detect",
        name="fabriq_defect_detection",
        exist_ok=True,
        patience=20,
        save=True,
        plots=True,
        verbose=True
    )
    
    print("\n✅ Training completed!")
    print(f"📊 Results saved in: runs/detect/fabriq_defect_detection/")
    
    return model, results

def validate_model(model):
    """Validate the trained detection model"""
    print("\n🔍 Validating model...")
    
    yaml_path = Path("fabriq_detection_data.yaml")
    results = model.val(data=str(yaml_path))
    
    print("\n📈 Validation Results:")
    print(f"   mAP50: {results.box.map50:.4f}")
    print(f"   mAP50-95: {results.box.map:.4f}")
    
    return results

if __name__ == "__main__":
    print("=" * 60)
    print("FabrIQ Fabric Defect Detection - YOLOv8 Object Detection Training")
    print("=" * 60)
    
    # Train detection model
    model, results = train_detection_model()
    
    if model is not None:
        # Validate
        validate_model(model)
        
        print("\n" + "=" * 60)
        print("🎉 Training Pipeline Complete!")
        print("=" * 60)
        print("\n💡 Next Steps:")
        print("   1. Check results in: runs/detect/fabriq_defect_detection/")
        print("   2. Use the best.pt model for inference with bounding boxes")
        print("   3. Test on new images using inference.py")



FabrIQ Fabric Defect Detection - YOLOv8 Object Detection Training

🚀 Starting YOLOv8 Object Detection Training...
📁 Preparing dataset structure for YOLO object detection...
   Copying train images...


SameFileError: PosixPath('/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset/train/images/FabrIQ_oil_lines_00092.jpg') and PosixPath('/kaggle/input/fabric/FabrIQ_YOLO_Detection_Dataset/train/images/FabrIQ_oil_lines_00092.jpg') are the same file

### Option B: Detection (Bounding Boxes)


In [21]:
# Create pseudo-annotations for detection
from pathlib import Path
import shutil

DATASET_PATH = Path('/kaggle/input/fabric-final-dataset/FabrIQ_Final_Dataset')
OUTPUT_PATH = Path('/kaggle/working/FabrIQ_YOLO_Detection_Dataset')

CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

print("📝 Creating pseudo-annotations...")

# Create structure
for split in ['train', 'val']:
    (OUTPUT_PATH / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_PATH / split / 'labels').mkdir(parents=True, exist_ok=True)

total_annotations = 0

for split in ['train', 'val']:
    source_dir = DATASET_PATH / split / 'images'
    count = 0
    
    for img_file in source_dir.glob("*.jpg"):
        filename = img_file.stem
        parts = filename.split('_')
        
        if len(parts) >= 2 and parts[0] == 'FabrIQ':
            class_parts = parts[1:-1]
            class_name = '_'.join(class_parts)
            class_name_with_spaces = class_name.replace('_', ' ')
            
            if class_name_with_spaces in CLASSES:
                # Copy image
                dest_img = OUTPUT_PATH / split / 'images' / img_file.name
                shutil.copy2(img_file, dest_img)
                
                # Create annotation (full image box)
                class_id = CLASSES.index(class_name_with_spaces)
                annotation = f"{class_id} 0.5 0.5 1.0 1.0\n"
                
                label_file = OUTPUT_PATH / split / 'labels' / f"{img_file.stem}.txt"
                with open(label_file, 'w') as f:
                    f.write(annotation)
                
                count += 1
                total_annotations += 1
    
    print(f"✅ {split}: {count} annotations created")

print(f"\n✅ Total: {total_annotations} annotations")
print(f"📁 Dataset ready at: {OUTPUT_PATH}")


📝 Creating pseudo-annotations...
✅ train: 22972 annotations created
✅ val: 5757 annotations created

✅ Total: 28729 annotations
📁 Dataset ready at: /kaggle/working/FabrIQ_YOLO_Detection_Dataset


In [22]:
# Create data.yaml for detection
import yaml

CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

data_config = {
    'path': '/kaggle/working/FabrIQ_YOLO_Detection_Dataset',
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CLASSES),
    'names': {i: cls for i, cls in enumerate(CLASSES)}
}

yaml_path = Path('/kaggle/working/fabriq_detection_data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Created data.yaml: {yaml_path}")


✅ Created data.yaml: /kaggle/working/fabriq_detection_data.yaml


In [34]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/kaggle/working/fabriq_detection_data.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    amp=False,
    name="fabriq_detection"
)


Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/fabriq_detection_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fabriq_detection2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

error: OpenCV(4.12.0) :-1: error: (-5:Bad argument) in function 'imdecode'
> Overload resolution failed:
>  - buf is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'buf'


In [38]:
import os

img_dirs = [
    "/kaggle/working/FabrIQ_YOLO_Detection_Dataset/train/images",
    "/kaggle/working/FabrIQ_YOLO_Detection_Dataset/val/images",
]

for d in img_dirs:
    for f in os.listdir(d):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            print("NON-IMAGE FILE:", os.path.join(d, f))
for d in img_dirs:
    for f in os.listdir(d):
        p = os.path.join(d, f)
        if os.path.islink(p):
            print("SYMLINK FOUND:", p)


In [24]:
import os
import cv2
from pathlib import Path

def clean_dataset():
    dataset_path = '/kaggle/working/FabrIQ_YOLO_Detection_Dataset'
    print(f"🧹 Scanning {dataset_path} for corrupt images...")
    
    removed = 0
    for file_path in Path(dataset_path).rglob('*'):
        if file_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
            # Check 1: Is file size 0?
            if file_path.stat().st_size == 0:
                print(f"   🗑️ Deleting empty file: {file_path.name}")
                os.remove(file_path)
                removed += 1
                continue
                
            # Check 2: Can OpenCV read it?
            try:
                img = cv2.imread(str(file_path))
                if img is None:
                    raise ValueError("Image is None")
            except Exception:
                print(f"   🗑️ Deleting corrupt file: {file_path.name}")
                os.remove(file_path)
                removed += 1

    print(f"\n✅ Scan Complete. Removed {removed} bad images.")

clean_dataset()

🧹 Scanning /kaggle/working/FabrIQ_YOLO_Detection_Dataset for corrupt images...

✅ Scan Complete. Removed 0 bad images.


In [15]:
import os
import shutil

# 1. Rename the folder to match what YOLO expects
current_name = '/kaggle/working/FabrIQ_Final_Dataset'
target_name = '/kaggle/working/FabrIQ_YOLO_Detection_Dataset'

if os.path.exists(current_name):
    print(f"🔄 Renaming '{current_name}' to '{target_name}'...")
    os.rename(current_name, target_name)
    print("✅ Folder Renamed Successfully!")
elif os.path.exists(target_name):
    print("✅ Folder name is already correct.")
else:
    print("❌ CRITICAL ERROR: Could not find the dataset folder. Did the 'organize' script finish running?")

# 2. Verify images are actually inside
train_images_path = os.path.join(target_name, 'train', 'images')
if os.path.exists(train_images_path):
    count = len(os.listdir(train_images_path))
    print(f"📊 Verification: Found {count} images in the training folder.")
else:
    print("⚠️ Warning: Training folder is still empty or missing!")

✅ Folder name is already correct.
📊 Verification: Found 0 images in the training folder.


In [10]:
# run this in a cell, then restart the runtime (Runtime > Restart Runtime)
!pip install ultralytics==8.0.196

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.1/631.1 kB 12.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.3.235
    Uninstalling ultralytics-8.3.235:
      Successfully uninstalled ultralytics-8.3.235


## Step 5: Download Results


In [ ]:
# Download trained model and results
from google.colab import files
from pathlib import Path
import zipfile

# Create zip file with results
results_dir = Path('/content/runs')
zip_path = Path('/content/fabriq_results.zip')

if results_dir.exists():
    print("📦 Creating zip file...")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in results_dir.rglob('*'):
            if file_path.is_file():
                zipf.write(file_path, file_path.relative_to(results_dir.parent))
    
    print(f"✅ Created: {zip_path}")
    print("\n📥 Downloading...")
    files.download(str(zip_path))
    
    # Also download best model separately
    best_model = Path('/content/runs/classify/fabriq_classification/weights/best.pt')
    if best_model.exists():
        files.download(str(best_model))
        print("✅ Best model downloaded")
else:
    print("⚠️ Results directory not found")


In [3]:
"""
YOLOv8 Object Detection Training Script for FabrIQ Fabric Defect Detection
This script trains a detection model that can predict bounding boxes around defects.
Note: Requires YOLO-format annotation files (.txt) with bounding boxes.
"""

import os
from pathlib import Path
from ultralytics import YOLO
import yaml

# Configuration
DATASET_PATH = Path("FabrIQ_Final_Dataset")
MODEL_TYPE = "yolov8n.pt"  # Detection model
# Auto-detect device
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":
        EPOCHS = 30
        IMG_SIZE = 416  # Smaller for CPU
        BATCH_SIZE = 4
    else:
        EPOCHS = 100
        IMG_SIZE = 640
        BATCH_SIZE = 16
except:
    DEVICE = "cpu"
    EPOCHS = 10
    IMG_SIZE = 416
    BATCH_SIZE = 4

# 20 Target Classes
CLASSES = [
    'bad needle line', 'creases', 'double kunda', 'end out', 'fluff knit',
    'fly yarn', 'knit hole', 'lycra short', 'mis pattern', 'mix yarn',
    'normal', 'oil lines', 'oil spot', 'press off', 'pulling thread',
    'run of needle', 'single kunda', 'sinker line', 'tight feeder', 'yarn variation'
]

def create_data_yaml():
    """Create data.yaml configuration file for YOLO detection"""
    dataset_path = Path("FabrIQ_YOLO_Detection_Dataset")
    
    data_config = {
        'path': str(dataset_path.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'nc': len(CLASSES),
        'names': {i: cls for i, cls in enumerate(CLASSES)}
    }
    
    yaml_path = Path("fabriq_detection_data.yaml")
    with open(yaml_path, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
    
    print(f"✅ Created data configuration: {yaml_path}")
    return yaml_path

def prepare_detection_dataset():
    """Prepare dataset structure for YOLO object detection"""
    print("📁 Preparing dataset structure for YOLO object detection...")
    
    yolo_dataset = Path("FabrIQ_YOLO_Detection_Dataset")
    
    # Create directory structure
    for split in ['train', 'val']:
        (yolo_dataset / split / 'images').mkdir(parents=True, exist_ok=True)
        (yolo_dataset / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Copy images
    for split in ['train', 'val']:
        source_dir = DATASET_PATH / split / 'images'
        
        if not source_dir.exists():
            print(f"⚠️ Warning: {source_dir} does not exist!")
            continue
        
        print(f"   Copying {split} images...")
        image_count = 0
        
        for img_file in source_dir.glob("*.jpg"):
            dest_img = yolo_dataset / split / 'images' / img_file.name
            import shutil
            # Avoid SameFileError if source and destination are identical (e.g., dataset already copied)
            if dest_img.exists():
                continue
            if Path(img_file).resolve() == dest_img.resolve():
                continue
            shutil.copy2(img_file, dest_img)
            image_count += 1
        
        print(f"   ✅ {split}: {image_count} images copied")
        print(f"   ⚠️ Note: You need to create annotation files (.txt) in {yolo_dataset / split / 'labels'}/")
        print(f"      Format: class_id center_x center_y width height (normalized 0-1)")
    
    return yolo_dataset

def train_detection_model():
    """Train YOLOv8 object detection model"""
    print("\n🚀 Starting YOLOv8 Object Detection Training...")
    
    # Prepare dataset
    dataset_path = prepare_detection_dataset()
    
    # Create data.yaml
    yaml_path = create_data_yaml()
    
    # Check if annotation files exist
    train_labels = list((dataset_path / 'train' / 'labels').glob('*.txt'))
    val_labels = list((dataset_path / 'val' / 'labels').glob('*.txt'))
    
    if len(train_labels) == 0:
        print("\n⚠️ WARNING: No annotation files found!")
        print("   For object detection, you need bounding box annotations.")
        print("   Run create_annotations.py or manually annotate images using tools like:")
        print("   - LabelImg (https://github.com/tzutalin/labelImg)")
        print("   - Roboflow (https://roboflow.com/)")
        print("   - CVAT (https://cvat.org/)")
        return None, None
    
    print(f"\n   Found {len(train_labels)} train annotations and {len(val_labels)} val annotations")
    
    # Initialize model
    print(f"\n📦 Loading model: {MODEL_TYPE}")
    model = YOLO(MODEL_TYPE)
    
    # Train the model
    print(f"\n🎯 Training Configuration:")
    print(f"   Dataset: {dataset_path}")
    print(f"   Config: {yaml_path}")
    print(f"   Epochs: {EPOCHS}")
    print(f"   Image Size: {IMG_SIZE}")
    print(f"   Batch Size: {BATCH_SIZE}")
    print(f"   Device: {DEVICE}")
    print(f"   Classes: {len(CLASSES)}")
    
    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project="runs/detect",
        name="fabriq_defect_detection",
        exist_ok=True,
        patience=20,
        save=True,
        plots=True,
        verbose=True
    )
    
    print("\n✅ Training completed!")
    print(f"📊 Results saved in: runs/detect/fabriq_defect_detection/")
    
    return model, results

def validate_model(model):
    """Validate the trained detection model"""
    print("\n🔍 Validating model...")
    
    yaml_path = Path("fabriq_detection_data.yaml")
    results = model.val(data=str(yaml_path))
    
    print("\n📈 Validation Results:")
    print(f"   mAP50: {results.box.map50:.4f}")
    print(f"   mAP50-95: {results.box.map:.4f}")
    
    return results

if __name__ == "__main__":
    print("=" * 60)
    print("FabrIQ Fabric Defect Detection - YOLOv8 Object Detection Training")
    print("=" * 60)
    
    # Train detection model
    model, results = train_detection_model()
    
    if model is not None:
        # Validate
        validate_model(model)
        
        print("\n" + "=" * 60)
        print("🎉 Training Pipeline Complete!")
        print("=" * 60)
        print("\n💡 Next Steps:")
        print("   1. Check results in: runs/detect/fabriq_defect_detection/")
        print("   2. Use the best.pt model for inference with bounding boxes")
        print("   3. Test on new images using inference.py")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FabrIQ Fabric Defect Detection - YOLOv8 Object Detection Training

🚀 Starting YOLOv8 Object Detection Training...
📁 Preparing dataset structure for YOLO object detection...
⚠️ Warning: FabrIQ_Final_Dataset/train/images does not exist!
⚠️ Warning: FabrIQ_Final_Dataset/val/images does not exist!
✅ Created data configuration: fabriq_detection_data.yaml

⚠️ WARNING: No annotation files found!
   For object detection, you need bounding box annotations.
   Run create_annotations.py or manually annotate images using tools like:
   - LabelImg (https://github.com/tzutalin/labelImg)
   - Roboflow (https://roboflow.com/)
   - CVAT (https://cvat.org/)


## 📊 Training Summary

After training completes:

1. **Check Results**: `/content/runs/classify/fabriq_classification/` or `/content/runs/detect/fabriq_detection/`
2. **Best Model**: `weights/best.pt`
3. **Training Curves**: `results.png`
4. **Confusion Matrix**: `confusion_matrix.png`

## 💡 Tips

- **GPU Time**: Training takes 30-60 minutes on Colab GPU
- **Save Progress**: Colab sessions disconnect after inactivity
- **Download Models**: Always download your trained models before disconnecting
- **Monitor**: Check training progress in the output cells

## 🎉 Done!

Your model is trained and ready to use!
